# BERTopic — Template para Novo Corpus

Copie este notebook para `notebooks/XX_bertopic_<nome>.ipynb` e substitua
os placeholders marcados com `# <<` para adaptar a um novo corpus.

## Pipeline (5 etapas)
1. **Pré-processamento** — stop words customizadas, CountVectorizer com acentos
2. **Embeddings** — Qwen3 0.6B via Ollama (1024d nativo, cacheados)
3. **Redução + Clustering** — UMAP 5D → HDBSCAN (leaf)
4. **Representação** — c-TF-IDF + MMR (diversity=0.2)
5. **Pós-processamento** — reduce_outliers → update_topics → nomeação LLM

## 6 Métricas avaliadas
Topic Diversity, Taxa de outliers, Coerência C_v, Coerência Semântica,
Exclusividade, FREX (Frequency × Exclusivity)

In [ ]:
# Descomente se necessário
# !pip install bertopic gensim sentence-transformers umap-learn hdbscan httpx nest_asyncio itables wordcloud -q
# !python -m spacy download pt_core_news_lg -q

In [ ]:
import yaml, re, asyncio, httpx, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text as sk_text
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from scipy.stats import rankdata
import spacy

# ── CONFIGURAÇÃO — edite aqui para cada corpus ──────────────────────────
CORPUS      = "social"            # << nome do corpus no params.yaml
LANGUAGE    = "multilingual"      # << "multilingual" (PT-BR) ou "english" (EN)
SPACY_MODEL = "pt_core_news_lg"   # << "pt_core_news_lg" (PT) ou "en_core_web_sm" (EN)
TOKEN_PATTERN = r"(?u)\b[a-zA-ZÀ-ÿ_]{2,}\b"  # << ajustar para EN se necessário
# ────────────────────────────────────────────────────────────────────────

with open("../configs/params.yaml", encoding="utf-8") as f:
    params = yaml.safe_load(f)

corpus_cfg  = params["corpora"][CORPUS]
TEXT_COL    = corpus_cfg["text_column"]                   # << chave real do params.yaml
ID_COL      = corpus_cfg.get("post_id_column", "post_id")  # chave real do params.yaml      # << social não define explicitamente
SUBDIR      = corpus_cfg.get("subdir", CORPUS)
SEED        = params.get("seed", 42)
np.random.seed(SEED)

bt_cfg      = params["bertopic"]
MODEL_NAME  = bt_cfg["embedding_model"]
DIMENSION   = bt_cfg.get("embedding_dimension", None)
SAFE_MODEL  = MODEL_NAME.replace("/", "_").replace(":", "_")
SUFFIX      = "1024d"

INPUT_DIR   = Path("../data/input") / SUBDIR
OUTPUT_DIR  = Path("../data/output") / SUBDIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_PATH  = INPUT_DIR / f"embeddings_{SAFE_MODEL}_{SUFFIX}.npy"

print(f"Corpus   : {CORPUS}  |  Idioma: {LANGUAGE}")
print(f"Modelo   : {MODEL_NAME}  |  Dim: {DIMENSION or '1024d nativo'}")
print(f"Cache    : {CACHE_PATH.name}")

In [ ]:
corpus_csv = INPUT_DIR / "corpus_limpo.csv"
if not corpus_csv.exists():
    raise FileNotFoundError(
        f"corpus_limpo.csv não encontrado: {corpus_csv}\n"
        f"Execute 01-preprocessing/notebooks/01_preprocessing.ipynb e copie o CSV para {INPUT_DIR}"
    )
df   = pd.read_csv(corpus_csv, encoding="utf-8")
docs = df[TEXT_COL].astype(str).tolist()

if not CACHE_PATH.exists():
    raise FileNotFoundError(
        f"Cache de embeddings não encontrado: {CACHE_PATH}\n"
        f"Execute 02-embeddings/notebooks/02_embeddings.ipynb e copie o .npy para {INPUT_DIR}"
    )
embeddings = np.load(CACHE_PATH)
assert embeddings.shape[0] == len(docs), "Desalinhamento corpus ↔ embeddings"
assert embeddings.shape[1] == 1024,      "Dimensão esperada: 1024d"

print(f"Docs       : {len(docs)}")
print(f"Embeddings : {embeddings.shape}  — 1024d OK")
print(f"\nExemplo: {docs[0][:120]}")

## Pré-processamento — CountVectorizer

O CountVectorizer do BERTopic gera as keywords (c-TF-IDF) — **não** os embeddings.
Os três ajustes críticos:

1. **Stop words customizadas** — NLTK base + termos de domínio específicos do corpus
2. **ngram_range=(1,2)** — habilita bigramas relevantes (ex: "saúde pública")
3. **token_pattern** — preserva acentos PT-BR (`À-ÿ`) e tokens de emojis demojizados (`_`)

In [ ]:
from nltk.corpus import stopwords
import nltk
nltk.download("stopwords", quiet=True)

# Stop words base por idioma
if LANGUAGE == "multilingual":
    base_sw = set(stopwords.words("portuguese"))
else:
    base_sw = set(sk_text.ENGLISH_STOP_WORDS)

# << Adicione stop words de domínio do corpus abaixo
STOPWORDS_DOMAIN = set([
    # Ex para social: "raquellyra", "governodepernambuco", "pernambuco"
    # Ex para AG News: "said", "new", "year"
])

# Stop words de emojis genéricos demojizados (PT-BR)
STOPWORDS_EMOJIS = set([
    "coração_roxo", "brilhos", "mãos_aplaudindo", "polegar_para_cima",
])

all_stop = base_sw | STOPWORDS_DOMAIN | STOPWORDS_EMOJIS

vectorizer = CountVectorizer(
    stop_words=list(all_stop),
    ngram_range=(1, 2),
    token_pattern=TOKEN_PATTERN,
    min_df=2,
)
print(f"Stop words total: {len(all_stop)}")

In [ ]:
umap_model = UMAP(
    n_components=5,
    n_neighbors=15,
    min_dist=0.0,
    metric="cosine",
    random_state=SEED,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=bt_cfg["hdbscan"]["min_cluster_size"],
    min_samples=bt_cfg["hdbscan"]["min_samples"],
    cluster_selection_method=bt_cfg["hdbscan"]["cluster_selection_method"],
    prediction_data=True,
)

representation_model = MaximalMarginalRelevance(diversity=bt_cfg["mmr_diversity"])

topic_model = BERTopic(
    language=LANGUAGE,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    representation_model=representation_model,
    min_topic_size=bt_cfg["min_topic_size"],
    calculate_probabilities=True,
    verbose=True,
)

t0 = time.time()
topics, probs = topic_model.fit_transform(docs, embeddings)
print(f"\nTreinamento: {time.time()-t0:.1f}s")
print(f"Tópicos encontrados: {topic_model.get_topic_info().shape[0] - 1} (+1 outlier)")
print(f"Outliers (-1): {sum(t == -1 for t in topics)} ({sum(t == -1 for t in topics)/len(topics):.1%})")

In [ ]:
# ORDEM CORRETA (B18):
#   reduce_outliers  → retorna novos assignments (NÃO atualiza o modelo)
#   update_topics    → sincroniza o modelo com os assignments sem outliers  ← obrigatório antes de reduce_topics
#   reduce_topics    → mescla os menores tópicos até nr_topics
#   update_topics    → atualiza representações c-TF-IDF + MMR após a redução
# Pular o 1º update_topics faz reduce_topics operar sobre assignments desatualizados.

# 1. Reatribuir outliers via c-TF-IDF
new_topics = topic_model.reduce_outliers(docs, topics, strategy="c-tf-idf")

# 2. Sincronizar modelo com assignments sem outliers
topic_model.update_topics(docs, topics=new_topics, vectorizer_model=vectorizer,
                           representation_model=representation_model)

# 3. Reduzir para K tópicos alvo
topic_model.reduce_topics(docs, nr_topics=bt_cfg["reduce_topics_nr"])
new_topics = topic_model.topics_

# 4. Atualizar representações c-TF-IDF + MMR com K final
topic_model.update_topics(docs, topics=new_topics, vectorizer_model=vectorizer,
                           representation_model=representation_model)
topics = new_topics
topic_info = topic_model.get_topic_info()

print(f"Tópicos após redução: {topic_info[topic_info.Topic != -1].shape[0]}")
print(f"Outliers remanescentes: {sum(t == -1 for t in topics)}")
topic_info.head(10)

In [ ]:
LLM_MODEL = params["llm"]["model"]  # "gemma4:e4b"
LLM_URL   = params["llm"]["base_url"]

# _strip_thinking importado de _helpers — versão mais robusta (extrai fallback
# de dentro do bloco <think> quando o modelo emite só thinking sem resposta).
# Este notebook assume execução a partir de 04-topic-modeling/notebooks/.
from _helpers import _strip_thinking

def name_topic(keywords, docs_sample, model=LLM_MODEL, base_url=LLM_URL):
    kw_str   = ", ".join(keywords[:10])
    docs_str = "\n".join([f"- {d[:200]}" for d in docs_sample[:2]])
    prompt   = (
        f"Você é especialista em análise de tópicos. "
        f"Palavras-chave: {kw_str}\n"
        f"Documentos representativos:\n{docs_str}\n\n"
        f"Gere um rótulo nominal conciso (3-5 palavras) para este tópico. "
        f"Responda apenas o rótulo, sem explicações."
    )
    try:
        with httpx.Client(timeout=30) as client:
            resp = client.post(f"{base_url}/api/generate",
                               json={"model": model, "prompt": prompt,
                                     "temperature": 0.2, "stream": False})
            raw  = resp.json()["response"].strip()
            name = _strip_thinking(raw).split("\n")[0]
            # Rejeitar se o nome é apenas lista de keywords
            if all(kw.lower() in name.lower() for kw in keywords[:3]):
                return ", ".join(keywords[:3])
            return name
    except Exception:
        return ", ".join(keywords[:3])

# Warm-up do LLM
with httpx.Client(timeout=10) as c:
    try:
        c.post(f"{LLM_URL}/api/generate",
               json={"model": LLM_MODEL, "prompt": "ok", "stream": False})
        print(f"LLM warm-up OK: {LLM_MODEL}")
    except Exception:
        print("AVISO: Ollama não disponível — nomes serão top-3 keywords")

rotulos = {}
valid_ids = [t for t in topic_info["Topic"].tolist() if t != -1]
for tid in valid_ids:
    kws  = [w for w, _ in topic_model.get_topic(tid)]
    reps = topic_model.get_representative_docs(tid)
    rotulos[tid] = name_topic(kws, reps)
    print(f"  T{tid}: {rotulos[tid]}")

topic_model.set_topic_labels(rotulos)

In [ ]:
# 1. Topic Diversity
def topic_diversity(model, top_k=10):
    valid = [t for t in model.get_topics() if t != -1]
    all_words = [w for t in valid for w, _ in model.get_topic(t)[:top_k]]
    return len(set(all_words)) / len(all_words) if all_words else 0

# 2. Taxa de outliers
outlier_rate = sum(t == -1 for t in topics) / len(topics)

# 3. Coerência C_v via gensim
stop_set = all_stop
tokenized = [
    [w for w in re.sub(r"[^a-zA-ZÀ-ÿ\s]", " ", doc.lower()).split()
     if w not in stop_set and len(w) > 2]
    for doc in docs
]
dictionary = Dictionary(tokenized)
topic_words = {t: [w for w, _ in topic_model.get_topic(t)[:20]]
               for t in valid_ids}
coherence_model = CoherenceModel(
    topics=list(topic_words.values()),
    texts=tokenized,
    dictionary=dictionary,
    coherence="c_v"
)
cv_score = coherence_model.get_coherence()

# 4. Coerência Semântica
embedder_sem = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
sem_scores = []
for tid in valid_ids:
    kws  = [w for w, _ in topic_model.get_topic(tid)[:10]]
    embs = embedder_sem.encode(kws)
    sims = cosine_similarity(embs)
    np.fill_diagonal(sims, 0)
    sem_scores.append(sims.sum() / (len(kws) * (len(kws) - 1)) if len(kws) > 1 else 0)
sem_mean = np.mean(sem_scores)

# 5. Exclusividade
ctfidf = topic_model.c_tf_idf_.toarray()
excl_scores = []
for i, tid in enumerate(valid_ids):
    row   = ctfidf[i]
    total = ctfidf.sum(axis=0)
    total = np.where(total == 0, 1e-9, total)
    excl  = (row / total)[row > 0].mean() if (row > 0).any() else 0
    excl_scores.append(excl)
excl_mean = np.mean(excl_scores)

# 6. FREX
frex_scores = []
for i, tid in enumerate(valid_ids):
    row     = ctfidf[i]
    freq_r  = rankdata(row) / len(row)
    col_sum = ctfidf.sum(axis=0)
    excl_r  = rankdata(row / np.where(col_sum == 0, 1, col_sum)) / len(row)
    frex    = (2 / (1 / (freq_r + 1e-9) + 1 / (excl_r + 1e-9)))[row > 0].mean()
    frex_scores.append(frex)
frex_mean = np.mean(frex_scores)

print("=" * 55)
print(f"  Tópicos válidos      : {len(valid_ids)}")
print(f"  Taxa de outliers     : {outlier_rate:.1%}")
print(f"  Topic Diversity      : {topic_diversity(topic_model):.4f}  (↑ melhor)")
print(f"  Coerência C_v        : {cv_score:.4f}              (↑ melhor)")
print(f"  Coer. Semântica      : {sem_mean:.4f}              (↑ melhor)")
print(f"  Exclusividade        : {excl_mean:.4f}              (↑ melhor)")
print(f"  FREX                 : {frex_mean:.4f}              (↑ melhor)")
print("=" * 55)

In [ ]:
results = pd.DataFrame({
    ID_COL:                   df[ID_COL].values,
    "topic_id":               topics,
    "topic_name":             [rotulos.get(t, f"Topic_{t}") for t in topics],
    "topic_prob_distribution": [str(list(p.round(4))) for p in probs],
    "topic_type":             "semantic",
    "granularity":            "unit",
})
out_path = OUTPUT_DIR / "bertopic_results.csv"
results.to_csv(out_path, index=False, encoding="utf-8")
print(f"Salvo: {out_path}  ({results.shape})")
results.head(5)

In [ ]:
# Distribuição de docs por tópico
counts = pd.Series(topics).value_counts().sort_index()
counts_valid = counts[counts.index != -1]

plt.figure(figsize=(12, 4))
plt.bar(counts_valid.index, counts_valid.values, color="steelblue")
plt.xlabel("Tópico")
plt.ylabel("Documentos")
plt.title(f"Distribuição de documentos por tópico — {CORPUS}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "bertopic_distribution.png", dpi=150)
plt.show()

# Barchart interativo (requer plotly)
try:
    fig = topic_model.visualize_barchart(top_n_topics=min(12, len(valid_ids)), n_words=10)
    fig.show()
except Exception as e:
    print(f"visualize_barchart indisponível: {e}")